# 🌍 Energy Learning Hub — RAG-Powered Knowledge System

A **Retrieval-Augmented Generation (RAG)** platform for interactive energy education featuring:

- **Tiered FAQ pathways** (Beginner → Intermediate → Advanced)
- **Jump-start answers** with deep-dive RAG retrieval from your PDFs
- **Gemini ↔ Qwen auto-fallback** for resilient LLM responses
- **Gemini API-based embeddings** — no local PyTorch required
- **Smart vector store** — loads from disk if already built (skips re-embedding)
- **Interactive Gradio UI** with tabbed layout: Ask & Compare, Knowledge Base, About

---

### How to Run
1. **Section 1** — Mount Drive & install packages
2. **Section 2** — Load and chunk your energy PDFs
3. **Section 3** — Build (or reload) the vector store
4. **Section 4** — Configure LLMs (Gemini + Qwen fallback)
5. **Section 5** — Build the RAG pipeline
6. **Section 6** — Load FAQ knowledge base
7. **Section 7** — Launch the interactive hub

---
## 1. Mount Google Drive & Install Packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Install required packages ---
!pip install --quiet \\
    langchain \\
    langchain-community \\
    langchain-google-genai \\
    langchain-chroma \\
    langchain-openai \\
    pypdf \\
    gradio \\
    google-generativeai \\
    google-ai-generativelanguage

print("\n✅ All packages installed successfully!")

---
## 2. Load & Chunk Documents

In [ ]:
import os
import glob
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ------------------------------------------------------------------
# Point this to YOUR Google Drive folder containing energy PDFs
# ------------------------------------------------------------------
data_dir = "/content/drive/MyDrive/energylearninghub/data"

ingestion_log = []
first_chunk_previews = {}
all_docs = []

pdf_files = sorted(glob.glob(os.path.join(data_dir, "**/*.pdf"), recursive=True))

for pdf_path in pdf_files:
    filename = os.path.basename(pdf_path)
    filesize_kb = os.path.getsize(pdf_path) / 1024
    filesize_mb = filesize_kb / 1024
    try:
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
        for page in pages:
            page.metadata["source_file"] = filename
            page.metadata["page_number"] = page.metadata.get("page", "N/A")
            first_line = page.page_content.strip().split("\n")[0][:100]
            if any(kw in first_line.lower() for kw in ["chapter", "part", "section", "appendix"]):
                page.metadata["chapter"] = first_line
            else:
                page.metadata["chapter"] = "N/A"
        all_docs.extend(pages)
        size_str = f"{filesize_mb:.2f} MB" if filesize_mb >= 1 else f"{filesize_kb:.1f} KB"
        ingestion_log.append({"filename": filename, "filesize": size_str, "pages": len(pages), "status": "✅ Loaded"})
        print(f"  ✅ {filename}: {len(pages)} pages ({size_str})")
    except Exception as e:
        ingestion_log.append({"filename": filename, "filesize": "?", "pages": 0, "status": f"❌ {e}"})
        print(f"  ❌ {filename}: {e}")

print(f"\n📚 Documents loaded: {len(all_docs)} pages from {len(pdf_files)} file(s)")

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_documents(all_docs)
total_chunks = len(chunks)

for chunk in chunks:
    src = chunk.metadata.get("source_file", "unknown")
    if src not in first_chunk_previews:
        first_chunk_previews[src] = chunk.page_content[:300]

print(f"✅ Split into {total_chunks} chunks")
print(f"\n📄 Preview of first chunk:\n")
print(list(first_chunk_previews.values())[0][:300], "...")

---
## 3. Build (or Reload) the Vector Store

In [ ]:
import time
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

GEMINI_KEY = userdata.get("GEMINI_API_KEY")
CHROMA_DIR = "./chroma_db"

# Gemini API-based embeddings — no local PyTorch required
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_KEY
)

vector_store = None

# --- Try loading existing vector store from disk first ---
if os.path.isdir(CHROMA_DIR) and os.listdir(CHROMA_DIR):
    print("💾 Found existing ChromaDB on disk, loading...")
    try:
        vector_store = Chroma(
            collection_name="energy_hub",
            embedding_function=embeddings,
            persist_directory=CHROMA_DIR
        )
        existing_count = vector_store._collection.count()
        if existing_count > 0:
            total_vectors = existing_count
            print(f"✅ Loaded existing vector store: {total_vectors} vectors (skipping re-embedding)")
        else:
            vector_store = None
            print("⚠️ Existing store is empty, will rebuild...")
    except Exception as e:
        vector_store = None
        print(f"⚠️ Could not load existing store ({e}), will rebuild...")

In [ ]:
# --- Build vector store from chunks if not loaded from disk ---
# Uses batched embedding with retry to handle free-tier rate limits
# (Gemini free tier: ~100 req/min, 1000 req/day)

if vector_store is None:
    BATCH_SIZE = 80
    num_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"🧩 Embedding {total_chunks} chunks in {num_batches} batches of {BATCH_SIZE}...")

    for i in range(0, len(chunks), BATCH_SIZE):
        batch_num = i // BATCH_SIZE + 1
        batch = chunks[i : i + BATCH_SIZE]
        print(f"📦 Batch {batch_num}/{num_batches} ({len(batch)} chunks)...", end=" ")

        for attempt in range(5):
            try:
                if vector_store is None:
                    vector_store = Chroma.from_documents(
                        documents=batch,
                        collection_name="energy_hub",
                        embedding=embeddings,
                        persist_directory=CHROMA_DIR
                    )
                else:
                    vector_store.add_documents(batch)
                print("✅")
                break
            except Exception as e:
                if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                    wait = 60 * (2 ** attempt)
                    print(f"\n⚠️  Rate limited, retrying in {wait}s (attempt {attempt+1}/5)...")
                    time.sleep(wait)
                else:
                    raise

        if i + BATCH_SIZE < len(chunks):
            print("⏳ Pausing 60s for rate limit...")
            time.sleep(60)

    import gc
    del all_docs, chunks
    gc.collect()
    print("\n🧹 Freed intermediate data from memory.")

total_vectors = vector_store._collection.count()
print(f"\n✅ Vector store ready: {total_vectors} vectors")

---
## 4. Configure LLMs (Gemini + Qwen Fallback)

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

# ------------------------------------------------------------------
# Load API keys from Colab Secrets (Key icon in left sidebar)
# Required secrets: GEMINI_API_KEY, HF_TOKEN
# ------------------------------------------------------------------
GEMINI_KEY = userdata.get("GEMINI_API_KEY")
HF_TOKEN   = userdata.get("HF_TOKEN")

os.environ["GEMINI_API_KEY"]          = GEMINI_KEY
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

# Primary LLM: Gemini 2.5 Flash
gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
    google_api_key=GEMINI_KEY
)

# Fallback LLM: Qwen 2.5 via HuggingFace's OpenAI-compatible Inference API
# (No torch/transformers needed — just an HTTP call)
qwen_llm = ChatOpenAI(
    model="Qwen/Qwen2.5-7B-Instruct",
    base_url="https://api-inference.huggingface.co/v1/",
    api_key=HF_TOKEN,
    temperature=0.1,
    max_tokens=1024,
)

# Resilient chain: auto-switches to Qwen if Gemini hits quota
llm_with_fallback = gemini_llm.with_fallbacks([qwen_llm])

print("✅ LLMs configured")
print("   Primary:  Gemini 2.5 Flash")
print("   Fallback: Qwen 2.5-7B-Instruct (HuggingFace Inference API — no local torch)")

---
## 5. Build the RAG Pipeline

In [ ]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# ------------------------------------------------------------------
# Energy-education-focused RAG prompt
# Now includes source citation instruction
# ------------------------------------------------------------------
template = """You are an expert energy educator for a learning hub that teaches
newcomers about oil, gas, and the broader energy sector. Answer the question
using ONLY the provided context from the knowledge base.

Guidelines:
- Use clear, accessible language appropriate for the learner's level
- Include specific data, examples, or case studies from the context when available
- If the context doesn't contain enough information, say so honestly
- ALWAYS cite your sources using the format [Source: filename, Page X]
- End with a suggestion for what to explore next

Context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# Retriever: top-10 chunks
retriever = vector_store.as_retriever(search_kwargs={"k": 10})

# Build the RAG chain with fallback-enabled LLM
qa_chain = (
    {
        "context": itemgetter("query") | retriever,
        "question": itemgetter("query")
    }
    | prompt
    | llm_with_fallback
    | StrOutputParser()
)

print("✅ RAG pipeline ready!")
print(f"   Retriever: top-10 chunks")
print(f"   Vector store: {vector_store._collection.count()} vectors")

### Quick Pipeline Test

In [ ]:
from langchain_core.messages import HumanMessage

query = "What are the components of crude oil?"

# 1. RAG answer (grounded in your PDFs)
print("--- 📚 RAG Answer (Context-Aware) ---")
try:
    rag_response = qa_chain.invoke({"query": query})
    print(rag_response)
except Exception as e:
    print(f"RAG Error: {e}")

print("\n" + "="*60 + "\n")

# 2. LLM-only answer (general knowledge, for comparison)
print("--- 🗣️ LLM-only Answer (General Knowledge) ---")
try:
    raw_response = llm_with_fallback.invoke([HumanMessage(content=query)])
    print(raw_response.content)
except Exception as e:
    print(f"LLM Error: {e}")

---
## 6. FAQ Knowledge Base — Tiered Learning Pathways

In [ ]:
# ==================================================================
# Tiered FAQ Database
# Maps: FAQs → Jump-start answers → Suggested sources → Levels
# ==================================================================

FAQ_DATABASE = {
    "beginner": {
        "label": "🟢 Beginner — Foundations",
        "description": "Start here if you're new to energy concepts.",
        "categories": {
            "Energy Basics ⚡": [
                {"q": "What is the difference between oil, natural gas, and coal?", "jumpstart": "Oil is a liquid fossil fuel used mainly for transportation and petrochemicals. Natural gas is gaseous, often used for heating and electricity. Coal is solid, historically used for power generation and steelmaking. Each has different carbon intensity and energy density.", "sources": ["Introductory energy textbooks", "IEA/EIA primers"]},
                {"q": "How is energy measured (joules, BTUs, kWh)?", "jumpstart": "Energy is measured in joules (J), British Thermal Units (BTUs), or kilowatt-hours (kWh). For example, 1 kWh powers a 100-watt light bulb for 10 hours.", "sources": ["General science references", "EIA energy glossary"]},
                {"q": "What are primary vs. secondary energy sources?", "jumpstart": "Primary sources (coal, crude oil, sunlight) exist in nature. Secondary sources (electricity, gasoline) are converted from primary sources for practical use.", "sources": ["Introductory energy textbooks"]}
            ],
            "Energy Systems 🔌": [
                {"q": "How is electricity generated from fossil fuels?", "jumpstart": "Fossil fuels are burned to heat water, producing steam that spins turbines connected to generators. This mechanical energy is converted into electricity.", "sources": ["Power systems textbooks", "EIA resources"]},
                {"q": "What are upstream, midstream, and downstream operations?", "jumpstart": "Upstream covers exploration and production. Midstream involves transportation and storage (pipelines, shipping). Downstream refers to refining and distribution into usable products like gasoline.", "sources": ["Industry primers", "SPE resources"]},
                {"q": "What are renewable energy sources?", "jumpstart": "Renewables include solar, wind, hydro, and geothermal. They rely on naturally replenished resources and generally produce lower greenhouse gas emissions.", "sources": ["IRENA reports", "Introductory energy textbooks"]}
            ],
            "Careers & Skills 🎓": [
                {"q": "What skills are needed to work in the energy industry?", "jumpstart": "Technical skills (engineering, geology, data analysis), business skills (economics, project management), and increasingly sustainability knowledge are valuable.", "sources": ["SPE career guides", "AAPG resources"]}
            ]
        },
        "next_steps": {"Want technical detail?": "intermediate", "Want policy & global view?": "advanced"}
    },
    "intermediate": {
        "label": "🟡 Intermediate — Industry Insights",
        "description": "For learners who understand the basics and want to explore operations and markets.",
        "categories": {
            "Resources & Operations 🛢️": [
                {"q": "What are proven reserves and how are they estimated?", "jumpstart": "Proven reserves are quantities of oil or gas that geological and engineering data show can be recovered under current economic and operating conditions.", "sources": ["Petroleum geology textbooks", "SPE Reserves Definitions"]},
                {"q": "What is the difference between baseload and peak load power?", "jumpstart": "Baseload power is the steady supply needed 24/7 (often from coal, nuclear, or hydro). Peak load power meets demand spikes (often from gas turbines or renewables with storage).", "sources": ["Electricity markets textbooks", "Grid operations guides"]}
            ],
            "Markets & Economics 📈": [
                {"q": "How are oil prices determined?", "jumpstart": "Prices are influenced by supply and demand, geopolitical events, OPEC decisions, production costs, and futures trading. They can be highly volatile.", "sources": ["BP Statistical Review", "Market analysis reports"]},
                {"q": "What is the role of OPEC in global energy markets?", "jumpstart": "OPEC coordinates oil production policies among member states to influence global supply and stabilize prices.", "sources": ["OPEC annual reports", "Energy outlook publications"]}
            ],
            "Sustainability & Security 🌱": [
                {"q": "What is carbon capture and storage (CCS)?", "jumpstart": "CCS captures CO2 emissions from power plants or industrial sites and stores them underground to reduce atmospheric emissions.", "sources": ["CCS textbooks", "IEA CCS reports"]},
                {"q": "What is energy security and why is it important?", "jumpstart": "Energy security means ensuring reliable, affordable access to energy through diversifying sources, securing supply chains, and reducing import dependence.", "sources": ["IEA energy security publications", "Policy reports"]},
                {"q": "Which professional certifications are useful?", "jumpstart": "Certifications like PMP (Project Management), CFA (energy finance), or specialized petroleum engineering courses can boost career prospects.", "sources": ["Professional certification syllabi", "SPE Handbook"]}
            ]
        },
        "next_steps": {"Want market & geopolitics?": "advanced", "Want technical engineering?": "advanced"}
    },
    "advanced": {
        "label": "🔵 Advanced — Strategic & Global Perspectives",
        "description": "For learners ready to engage with complex policy, geopolitics, and future trends.",
        "categories": {
            "Policy & Regulation 🏛️": [
                {"q": "How do international agreements (like the Paris Accord) shape energy policy?", "jumpstart": "Agreements like the Paris Accord push countries to reduce emissions, invest in renewables, and set long-term climate targets that reshape national energy strategies.", "sources": ["UNFCCC publications", "IEA policy reports"]},
                {"q": "What are net-zero targets and how do they affect the energy transition?", "jumpstart": "Net-zero means balancing CO2 emissions with removals. These targets drive investment in renewables, CCS, and efficiency improvements across the energy sector.", "sources": ["World Bank reports", "IEA Net Zero Roadmap"]}
            ],
            "Geopolitics & Strategy 🌐": [
                {"q": "How do geopolitics influence oil and gas markets?", "jumpstart": "Political tensions, sanctions, trade agreements, and regional conflicts can disrupt supply chains, shift trade flows, and cause price volatility in global energy markets.", "sources": ["Energy geopolitics texts", "The Prize by Daniel Yergin"]},
                {"q": "What is the impact of subsidies and regulation on energy markets?", "jumpstart": "Subsidies can lower consumer costs but distort markets. Regulations set safety and environmental standards. Together, they shape investment decisions and energy mix.", "sources": ["Energy economics journals", "World Bank policy papers"]}
            ],
            "Future of Energy 🚀": [
                {"q": "How does the energy transition affect traditional oil and gas companies?", "jumpstart": "Many companies are diversifying into renewables, hydrogen, and CCS. The transition creates both risks (stranded assets) and opportunities (new markets).", "sources": ["Corporate sustainability reports", "IEA World Energy Outlook"]},
                {"q": "How do grids and transmission systems work at scale?", "jumpstart": "Power grids balance generation and demand in real-time across vast networks. Modern grids integrate renewables, storage, and smart technology for reliability.", "sources": ["Power systems engineering texts", "Grid integration studies"]},
                {"q": "What are the major career pathways in renewables vs. fossil fuels?", "jumpstart": "Fossil fuel careers focus on geology, drilling, and refining. Renewable careers span solar/wind engineering, grid integration, policy, and energy storage. Many skills transfer between sectors.", "sources": ["Workforce trend studies", "Professional association guides"]}
            ]
        },
        "next_steps": {}
    }
}

# Summary
print("✅ FAQ Knowledge Base loaded!\n")
for level, data in FAQ_DATABASE.items():
    total_qs = sum(len(qs) for qs in data["categories"].values())
    print(f"   {data['label']}: {total_qs} FAQs across {len(data['categories'])} categories")

---
## 7. Launch the Interactive Hub (Gradio)

In [ ]:
import gradio as gr
from langchain_core.messages import HumanMessage

# ==================================================================
# Helper: Detect which LLM model answered
# ==================================================================
def get_model_source(response_obj):
    """Identify whether Gemini or Qwen answered based on response metadata."""
    metadata = getattr(response_obj, "response_metadata", {})
    if "safety_ratings" in metadata or "prompt_feedback" in metadata:
        return "Gemini 2.5 Flash"
    return "Qwen 2.5 (HuggingFace Fallback)"


# ==================================================================
# Helper: Find matching FAQ jump-start answer
# ==================================================================
def find_jumpstart(question, level=None):
    """Search FAQ database for a matching jump-start answer."""
    q_lower = question.lower().strip()
    for lvl, data in FAQ_DATABASE.items():
        if level and lvl != level:
            continue
        for cat, faqs in data["categories"].items():
            for faq in faqs:
                if faq["q"].lower().strip() == q_lower:
                    return {"answer": faq["jumpstart"], "sources": faq["sources"], "level": lvl, "category": cat}
    return None


# ==================================================================
# Helper: Format FAQ list for sidebar
# ==================================================================
def get_faq_list(level_key):
    """Return formatted FAQ list for a given level."""
    if level_key not in FAQ_DATABASE:
        return "Select a level to see FAQs."
    data = FAQ_DATABASE[level_key]
    lines = ["### " + data['label'], "_" + data['description'] + "_"]
    for cat, faqs in data["categories"].items():
        lines.append("\n**" + cat + "**")
        for faq in faqs:
            lines.append("- " + faq['q'])
    if data["next_steps"]:
        lines.append("\n---\n**👉 Where to go next:**")
        for label, target in data["next_steps"].items():
            lines.append("- " + label + " → **" + target.capitalize() + "** level")
    return "\n".join(lines)


# ==================================================================
# Helper: Knowledge Base ingestion summary
# ==================================================================
def get_ingestion_summary():
    """Return a formatted summary of ingested documents and vector store stats."""
    if not ingestion_log:
        return "No PDFs found in the data directory."
    lines = ["### 📚 Ingested Documents\n", "| File | Size | Pages | Status |", "|------|------|-------|--------|"]
    for entry in ingestion_log:
        lines.append("| " + entry["filename"] + " | " + entry["filesize"] + " | " + str(entry["pages"]) + " | " + entry["status"] + " |")
    lines.append("\n**Total chunks:** " + str(total_chunks))
    lines.append("**Vector store:** " + str(total_vectors) + " vectors")
    if first_chunk_previews:
        lines.append("\n---\n### 🔍 First Chunk Preview\n")
        for src, preview in first_chunk_previews.items():
            lines.append("**" + src + ":**")
            lines.append("> " + preview[:250].replace("\n", " ") + "...\n")
    return "\n".join(lines)


# ==================================================================
# Core: Query the hub (jump-start + RAG deep dive + LLM comparison)
# ==================================================================
def query_hub(user_query, selected_level):
    level_map = {
        "All Levels": None,
        "🟢 Beginner": "beginner",
        "🟡 Intermediate": "intermediate",
        "🔵 Advanced": "advanced"
    }
    level_filter = level_map.get(selected_level)

    # --- Jump-start lookup ---
    js = find_jumpstart(user_query, level_filter)
    if js:
        jumpstart_md = (
            "📋 **Jump-Start Answer** (" + js['level'].capitalize() + " | " + js['category'] + ")"
            + "\n\n" + js['answer']
            + "\n\n📚 Suggested sources: " + ', '.join(js['sources'])
        )
    else:
        jumpstart_md = "_No matching FAQ — showing RAG retrieval only._"

    # --- RAG deep-dive ---
    try:
        rag_response = qa_chain.invoke({"query": user_query})
        rag_md = "📚 **RAG Deep-Dive Answer**\n\n" + str(rag_response)
    except Exception as e:
        rag_md = "❌ RAG Error: " + str(e)

    # --- Direct LLM comparison ---
    try:
        llm_response = llm_with_fallback.invoke([HumanMessage(content=user_query)])
        model_name = get_model_source(llm_response)
        llm_md = "🤖 [Source: " + model_name + "]\n\n" + llm_response.content
    except Exception as e:
        llm_md = "❌ LLM Error: " + str(e)

    return jumpstart_md, rag_md, llm_md


# ==================================================================
# Gradio UI — Tabbed layout matching app.py
# ==================================================================
with gr.Blocks(title="Energy Learning Hub") as demo:

    # --- Header ---
    gr.Markdown(
        "# 🌍 Energy Learning Hub\n"
        "**RAG-Powered Knowledge System** | Auto-Switching: Gemini ↔ Qwen | Tiered Learning Pathways"
    )

    with gr.Tabs():

        # ── Tab 1: Ask & Compare ─────────────────────────────────────
        with gr.TabItem("🔍 Ask & Compare"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 🧭 Learning Pathway")
                    level_selector = gr.Radio(
                        choices=["beginner", "intermediate", "advanced"],
                        value="beginner",
                        label="Select your level"
                    )
                    faq_display = gr.Markdown(value=get_faq_list("beginner"))

                with gr.Column(scale=2):
                    with gr.Row():
                        query_input = gr.Textbox(
                            label="Your question",
                            placeholder="e.g., What is the difference between oil, natural gas, and coal?",
                            lines=1,
                            scale=4
                        )
                        level_filter = gr.Dropdown(
                            choices=["All Levels", "🟢 Beginner", "🟡 Intermediate", "🔵 Advanced"],
                            value="All Levels",
                            label="Filter",
                            scale=1
                        )
                    submit_btn = gr.Button("🔍 Ask the Hub", variant="primary")
                    jumpstart_out = gr.Markdown(label="Jump-Start Answer")
                    with gr.Row():
                        rag_out = gr.Markdown(label="RAG Deep-Dive")
                        llm_out = gr.Markdown(label="LLM General Knowledge")

            level_selector.change(fn=get_faq_list, inputs=level_selector, outputs=faq_display)
            submit_btn.click(fn=query_hub, inputs=[query_input, level_filter], outputs=[jumpstart_out, rag_out, llm_out])
            query_input.submit(fn=query_hub, inputs=[query_input, level_filter], outputs=[jumpstart_out, rag_out, llm_out])

        # ── Tab 2: Knowledge Base ────────────────────────────────────
        with gr.TabItem("📚 Knowledge Base"):
            kb_display = gr.Markdown(value=get_ingestion_summary())
            refresh_btn = gr.Button("🔄 Refresh Status")
            refresh_btn.click(fn=get_ingestion_summary, inputs=None, outputs=kb_display)

        # ── Tab 3: About ─────────────────────────────────────────────
        with gr.TabItem("ℹ️ About"):
            gr.Markdown(
                "## Energy Learning Hub\n\n"
                "This platform helps newcomers learn about the oil, gas, and energy industry "
                "through a RAG-powered system grounded in authoritative PDF sources.\n\n"
                "**Current data source:** Oil 101 by Morgan Downey\n\n"
                "**LLMs:** Gemini 2.5 Flash (primary) + Qwen 2.5-7B (fallback)\n\n"
                "**Built by:** Derek Eng | 2026"
            )

print("✅ Gradio UI built. Launching...")
demo.launch(share=True, debug=True, theme=gr.themes.Soft())

---
## 8. Utility — Test Queries & Debug

In [ ]:
# ==================================================================
# Test individual queries outside the Gradio UI
# ==================================================================
test_queries = [
    "What is the difference between oil, natural gas, and coal?",
    "Describe NG trading strategies",
    "How do geopolitics influence oil and gas markets?",
    "What is carbon capture and storage?",
]

for query in test_queries:
    print(f"\n🤔 Question: {query}")
    print("-" * 60)

    js = find_jumpstart(query)
    if js:
        print(f"📋 Jump-Start ({js['level']}): {js['answer'][:150]}...")

    try:
        response = qa_chain.invoke({"query": query})
        print(f"📚 RAG: {response[:200]}...")
    except Exception as e:
        print(f"❌ RAG Error: {e}")

    print()

### Check Available Google Models

In [ ]:
import google.generativeai as genai

api_key = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=api_key)

print("Available models with generateContent support:\n")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(f"  • {m.name}")